In [1]:
import sys, re, json
from pathlib import Path
from difflib import SequenceMatcher
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT))
from src.common.config import PROCESSED, GOLD, TEMPLATES, ask_json
import pandas as pd

projects = sorted(p for p in PROCESSED.glob("*/*") if p.is_dir() and not p.name.startswith("_"))
print(len(projects), "processed projects")

10 processed projects


In [2]:
SECTIONS = {
    "document_control":         "Title, version, application, prepared/reviewed by, audience, purpose statement",
    "references":               "Source documents the plan is based on (BRD, architecture diagram, etc.)",
    "objectives":               "What testing must prove; quality objectives and measurable targets",
    "scope_in":                 "Features, modules and quality attributes in scope (by MVP)",
    "scope_out":                "Explicitly out of scope / deferred items (incl. later MVPs)",
    "assumptions_dependencies": "Test assumptions, constraints, dependencies on systems, data or teams",
    "test_strategy":            "Approach, test levels, test types and techniques",
    "functional_coverage":      "Functional features / modules to be tested and what is verified for each",
    "rbac_testing":             "Roles, permissions, access control and approval workflow testing",
    "security_compliance":      "Security, privacy, compliance testing",
    "ai_data_quality":          "AI / extraction / retrieval accuracy, data quality and Responsible AI validation",
    "nfr_performance":          "Performance, load, scalability targets and tests",
    "nfr_reliability":          "Availability, resilience, recovery, retry and failure handling tests",
    "nfr_accessibility_compat": "Accessibility and browser / device compatibility",
    "observability_audit":      "Logging, monitoring, alerting and audit-trail testing",
    "test_data_environment":    "Test environments and test data management",
    "entry_exit_criteria":      "Entry, exit, suspension and resumption criteria",
    "defect_management":        "Defect severity definitions, tracking tool and triage process",
    "risks":                    "Test / quality risks and mitigations",
    "deliverables_reporting":   "Test deliverables and reporting",
    "traceability_metrics":     "Requirement-to-test traceability and test metrics / KPIs",
    "schedule":                 "Test schedule, phases, milestones",
    "e2e_scenario":             "Representative end-to-end scenario(s)",
    "approvals":                "Sign-off / approvals",
}
REQUIRED = [k for k in SECTIONS if k not in ("references", "schedule", "approvals", "deliverables_reporting")]

schema = {"version": "1.0", "sections": [{"key": k, "description": d, "required": k in REQUIRED}
                                         for k, d in SECTIONS.items()]}
TEMPLATES.mkdir(parents=True, exist_ok=True)
(TEMPLATES / "test_plan_schema.json").write_text(json.dumps(schema, indent=2), encoding="utf-8")
print(len(SECTIONS), "canonical sections,", len(REQUIRED), "required")

24 canonical sections, 20 required


In [3]:
MAP_SYSTEM = ("You are a senior QA lead. You map headings of a software test plan to a fixed list of "
              "canonical section keys. Judge by the heading AND its content preview, not the heading alone. "
              "A sub-section usually belongs to the same key as its parent unless its content clearly fits another key "
              "(e.g. 'Security' under 'Non-Functional Testing' -> security_compliance). A merged section such as "
              "'In Scope / Out of Scope' gets BOTH keys (max 2). Use 'other' only if nothing fits.")

MAP_SCHEMA = {
    "type": "object", "additionalProperties": False, "required": ["mapping"],
    "properties": {"mapping": {"type": "array", "items": {
        "type": "object", "additionalProperties": False, "required": ["index", "keys"],
        "properties": {"index": {"type": "integer"},
                       "keys": {"type": "array", "items": {"type": "string", "enum": list(SECTIONS) + ["other"]}}}}}}}

def map_sections(secs):
    keys = "\n".join(f"- {k}: {d}" for k, d in SECTIONS.items())
    rows, parent = [], ""
    for i, s in enumerate(secs):
        if s["level"] == 1:
            parent = s["heading"]
        preview = re.sub(r"\s+", " ", s["text"])[:220]
        rows.append(f"[{i}] H{s['level']} {s['heading']} | parent: {parent if s['level'] > 1 else '-'} | {preview}")
    prompt = f"CANONICAL KEYS:\n{keys}\n\nTEST PLAN SECTIONS:\n" + "\n".join(rows) + \
             "\n\nReturn one mapping entry for EVERY section index."
    out = ask_json(MAP_SYSTEM, prompt, MAP_SCHEMA, "section_mapping", effort="medium")
    m = {e["index"]: (e["keys"][:2] or ["other"]) for e in out["mapping"]}
    return [m.get(i, ["other"]) for i in range(len(secs))]

# try on one validation plan and eyeball it
secs = json.loads((next(p for p in projects if p.name.startswith("P06")) / "testplan_sections.json").read_text(encoding="utf-8"))
keys = map_sections(secs)
pd.DataFrame({"level": [s["level"] for s in secs], "heading": [s["heading"] for s in secs], "keys": keys})

,level,heading,keys
0,0,(front matter),[document_control]
1,2,References,[references]
2,1,Test Objectives,[objectives]
3,1,In Scope / Out of Scope,"[scope_in, scope_out]"
4,3,Out of scope,[scope_out]
5,1,Test Assumptions,[assumptions_dependencies]
6,1,Testing Strategy,[test_strategy]
7,2,Test levels,[test_strategy]
8,2,Test techniques,[test_strategy]
9,1,Test Coverage – Functional,[functional_coverage]


In [4]:
def build_gold(proj):
    secs = json.loads((proj / "testplan_sections.json").read_text(encoding="utf-8"))
    keys = map_sections(secs)
    gold = {"project": proj.name, "split": proj.parent.name, "title": None,
            "sections": {k: {"present": False, "source_headings": [], "text": ""} for k in SECTIONS},
            "functional_features": [], "unmapped": []}
    for s, ks in zip(secs, keys):
        if s["level"] == 0:
            ks = ["document_control"]
        elif "document_control" in ks and gold["title"] is None:
            gold["title"] = s["heading"]
        if ks == ["other"]:
            gold["unmapped"].append(s["heading"]); continue
        for k in ks:
            if k == "other":
                continue
            g = gold["sections"][k]
            g["present"] = True
            g["source_headings"].append(s["heading"])
            block = (f"### {s['heading']}\n" if s["level"] > 1 else "") + s["text"]
            g["text"] = (g["text"] + "\n\n" + block).strip()
            if k == "functional_coverage" and s["level"] > 1:
                gold["functional_features"].append(s["heading"])
    return gold

GOLD.mkdir(parents=True, exist_ok=True)
rows = []
for proj in projects:
    out = GOLD / proj.parent.name / proj.name
    out.mkdir(parents=True, exist_ok=True)
    gold = build_gold(proj)
    (out / "gold_testplan.json").write_text(json.dumps(gold, indent=2, ensure_ascii=False), encoding="utf-8")
    present = [k for k, v in gold["sections"].items() if v["present"]]
    rows.append({"split": proj.parent.name, "project": proj.name,
                 "sections_present": len(present),
                 "required_missing": ", ".join(k for k in REQUIRED if k not in present),
                 "features": len(gold["functional_features"]),
                 "unmapped": ", ".join(gold["unmapped"])})
gold_df = pd.DataFrame(rows)
gold_df

,split,project,sections_present,required_missing,features,unmapped
0,test,P08_Warranty_Returns,23,,0,Non-Functional Requirements Testing
1,test,P09_Vendor_Compliance,22,,5,Non-Functional Requirements Testing
2,train,P00_Pricing_Tool_UserSample,19,"objectives, scope_in, nfr_performance",5,
3,train,P01_Invoice_Matching,21,,5,Non-Functional Requirements Testing
4,train,P02_Quote_Assistant,21,,0,
5,train,P03_Contract_Intelligence,22,objectives,0,
6,train,P04_Product_Data_Enrichment,22,,5,
7,train,P05_Freight_Claims,24,,0,
8,validation,P06_Rebate_Validation,23,,5,
9,validation,P07_Email_Order_Entry,22,,5,


In [5]:
CATEGORIES = ["scope_mvp1", "scope_mvp2", "out_of_scope", "functional", "business_rule", "nfr_performance",
              "nfr_availability", "security_access", "role", "integration", "data", "ai_quality",
              "compliance", "assumption", "dependency", "risk", "acceptance", "timeline", "architecture"]

FACT_SYSTEM = ("You are a senior QA lead preparing to write a test plan. Extract ATOMIC, TESTABLE facts "
               "from the requirement documents. One fact = one checkable statement (one number, one rule, one system, "
               "one role, one scope item). Keep exact numbers, units, thresholds, names of systems and roles. "
               "Do not merge facts, do not infer, do not add anything that is not written. "
               "source_quote must be copied character-for-character from the document (max ~25 words).")

FACT_SCHEMA = {
    "type": "object", "additionalProperties": False, "required": ["facts"],
    "properties": {"facts": {"type": "array", "items": {
        "type": "object", "additionalProperties": False,
        "required": ["category", "mvp", "fact", "source_quote", "source", "must_test"],
        "properties": {
            "category": {"type": "string", "enum": CATEGORIES},
            "mvp": {"type": "string", "enum": ["MVP1", "MVP2", "All", "NA"]},
            "fact": {"type": "string"},
            "source_quote": {"type": "string"},
            "source": {"type": "string", "enum": ["brd", "architecture"]},
            "must_test": {"type": "boolean"}}}}}}

def extract_facts(proj):
    brd = (proj / "brd.md").read_text(encoding="utf-8")
    arch = (proj / "architecture.md").read_text(encoding="utf-8")
    prompt = (f"<brd>\n{brd}\n</brd>\n\n<architecture>\n{arch}\n</architecture>\n\n"
              "Extract every testable fact. Typical count: 40-90. "
              "must_test=true if a test plan for MVP1 would be wrong or incomplete without it.")
    return ask_json(FACT_SYSTEM, prompt, FACT_SCHEMA, "brd_facts", effort="medium")["facts"], brd + "\n" + arch

In [6]:
def _norm(s):
    s = s.lower().replace("–", "-").replace("—", "-").replace("’", "'").replace("“", '"').replace("”", '"')
    s = re.sub(r"[|*#`]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def quote_score(quote, source):
    q, src = _norm(quote), _norm(source)
    if not q:
        return 0.0
    if q in src:
        return 1.0
    m = SequenceMatcher(None, src, q, autojunk=False).find_longest_match(0, len(src), 0, len(q))
    return m.size / len(q)

def verify(facts, source, threshold=0.8):
    for i, f in enumerate(facts, 1):
        f["id"] = f"F{i:03d}"
        f["quote_score"] = round(quote_score(f["source_quote"], source), 2)
        f["verified"] = f["quote_score"] >= threshold
    return facts

fact_rows = []
for proj in projects:
    out = GOLD / proj.parent.name / proj.name
    if (out / "facts.json").exists() and json.loads((out / "facts.json").read_text(encoding="utf-8"))["reviewed"]:
        print("skip (already reviewed):", proj.name); continue
    facts, source = extract_facts(proj)
    facts = verify(facts, source)
    good = [f for f in facts if f["verified"]]
    (out / "facts.json").write_text(json.dumps({"project": proj.name, "reviewed": False, "facts": good,
                                                "rejected": [f for f in facts if not f["verified"]]},
                                               indent=2, ensure_ascii=False), encoding="utf-8")
    fact_rows.append({"project": proj.name, "extracted": len(facts), "verified": len(good),
                      "rejected": len(facts) - len(good), "must_test": sum(f["must_test"] for f in good),
                      "mvp1": sum(f["mvp"] == "MVP1" for f in good)})
facts_df = pd.DataFrame(fact_rows)
facts_df

,project,extracted,verified,rejected,must_test,mvp1
0,P08_Warranty_Returns,82,82,0,65,43
1,P09_Vendor_Compliance,94,92,2,84,42
2,P00_Pricing_Tool_UserSample,128,128,0,61,50
3,P01_Invoice_Matching,70,69,1,58,23
4,P02_Quote_Assistant,92,91,1,70,44
5,P03_Contract_Intelligence,98,98,0,80,57
6,P04_Product_Data_Enrichment,82,81,1,60,47
7,P05_Freight_Claims,83,81,2,67,62
8,P06_Rebate_Validation,98,98,0,80,71
9,P07_Email_Order_Entry,69,69,0,56,35


In [7]:
review = []
for proj in projects:
    data = json.loads((GOLD / proj.parent.name / proj.name / "facts.json").read_text(encoding="utf-8"))
    for f in data["facts"]:
        review.append({"project": proj.name, "id": f["id"], "category": f["category"], "mvp": f["mvp"],
                       "must_test": f["must_test"], "fact": f["fact"], "source_quote": f["source_quote"], "keep": "Y"})
pd.DataFrame(review).to_csv(GOLD / "facts_review.csv", index=False, encoding="utf-8-sig")
print("Review file:", GOLD / "facts_review.csv", "|", len(review), "facts")

Review file: C:\Users\Dhruba\OneDrive\Desktop\test_agent\data\gold\facts_review.csv | 889 facts


In [8]:
rev = pd.read_csv(GOLD / "facts_review.csv", encoding="utf-8-sig")
for proj in projects:
    path = GOLD / proj.parent.name / proj.name / "facts.json"
    data = json.loads(path.read_text(encoding="utf-8"))
    r = rev[rev.project == proj.name].set_index("id")
    if r.empty:
        continue
    kept = []
    for f in data["facts"]:
        if f["id"] in r.index and str(r.loc[f["id"], "keep"]).strip().upper() == "Y":
            for col in ("category", "mvp", "fact"):
                f[col] = r.loc[f["id"], col]
            f["must_test"] = str(r.loc[f["id"], "must_test"]).strip().lower() in ("true", "1", "y", "yes")
            kept.append(f)
    data["facts"], data["reviewed"] = kept, True
    path.write_text(json.dumps(data, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
    print(proj.name.ljust(30), len(kept), "facts kept")

P08_Warranty_Returns           82 facts kept
P09_Vendor_Compliance          92 facts kept
P00_Pricing_Tool_UserSample    128 facts kept
P01_Invoice_Matching           69 facts kept
P02_Quote_Assistant            91 facts kept
P03_Contract_Intelligence      98 facts kept
P04_Product_Data_Enrichment    81 facts kept
P05_Freight_Claims             81 facts kept
P06_Rebate_Validation          98 facts kept
P07_Email_Order_Entry          69 facts kept
